In [ ]:
from google.colab import drive
import os
import pandas as pd

In [ ]:
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# Step 2: Define paths
generated_data_dir = "/content/drive/MyDrive/Colab Notebooks/AE_WGAN_NSL_KDD/Generated_NSL_KDD"
preprocessed_file_path = "/content/drive/MyDrive/Colab Notebooks/AE_WGAN_NSL_KDD/preprocessed_NSL_KDD.csv"

In [ ]:
# Step 3: Load the original preprocessed dataset
df_original = pd.read_csv(preprocessed_file_path)

# Step 4: Find all generated CSV files in the specified folder
csv_files = [f for f in os.listdir(generated_data_dir) if f.endswith(".csv")]

# Step 5: Load and merge all generated data files
df_generated_list = [pd.read_csv(os.path.join(generated_data_dir, file)) for file in csv_files]

# Step 6: Combine original and generated datasets
df_combined = pd.concat([df_original] + df_generated_list, ignore_index=True)

NameError: name 'preprocessed_file_path' is not defined

In [ ]:
# Step 7: Save the merged dataset back to Google Drive
output_path = os.path.join("/content/drive/MyDrive/Colab Notebooks/AE_WGAN_NSL_KDD/", "merged_dataset.csv")
df_combined.to_csv(output_path, index=False)

print(f"Merged dataset saved at: {output_path}")

Merged dataset saved at: /content/drive/MyDrive/Colab Notebooks/AE_WGAN_NSL_KDD/merged_dataset.csv


In [ ]:
df_merged=pd.read_csv("/content/drive/MyDrive/Colab Notebooks/AE_WGAN_NSL_KDD/merged_dataset.csv")

In [ ]:
df_merged.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 129087 entries, 0 to 129086
Data columns (total 25 columns):
 #   Column                       Non-Null Count   Dtype  
---  ------                       --------------   -----  
 0   protocol_type                129087 non-null  float64
 1   flag                         129087 non-null  float64
 2   land                         129087 non-null  float64
 3   wrong_fragment               129087 non-null  float64
 4   logged_in                    129087 non-null  float64
 5   count                        129087 non-null  float64
 6   srv_count                    129087 non-null  float64
 7   serror_rate                  129087 non-null  float64
 8   srv_serror_rate              129087 non-null  float64
 9   rerror_rate                  129087 non-null  float64
 10  srv_rerror_rate              129087 non-null  float64
 11  same_srv_rate                129087 non-null  float64
 12  diff_srv_rate                129087 non-null  float64
 13 

In [ ]:
print(df_merged['label'].value_counts())

label
normal             67343
neptune            41214
satan               3633
ipsweep             3599
portsweep           2931
smurf               2646
back                1912
teardrop            1784
warezclient         1780
nmap                1493
pod                  402
guess_passwd         106
buffer_overflow       60
warezmaster           40
land                  36
imap                  22
rootkit               20
loadmodule            18
ftp_write             16
multihop              14
phf                    8
perl                   6
spy                    4
Name: count, dtype: int64


In [ ]:
newlabeldf=df_merged['label'].replace({ 'normal' : 0, 'neptune' : 1 ,'back': 1, 'land': 1, 'pod': 1, 'smurf': 1, 'teardrop': 1,'mailbomb': 1, 'apache2': 1, 'processtable': 1, 'udpstorm': 1, 'worm': 1,
                           'ipsweep' : 2,'nmap' : 2,'portsweep' : 2,'satan' : 2,'mscan' : 2,'saint' : 2
                           ,'ftp_write': 3,'guess_passwd': 3,'imap': 3,'multihop': 3,'phf': 3,'spy': 3,'warezclient': 3,'warezmaster': 3,'sendmail': 3,'named': 3,'snmpgetattack': 3,'snmpguess': 3,'xlock': 3,'xsnoop': 3,'httptunnel': 3,
                           'buffer_overflow': 4,'loadmodule': 4,'perl': 4,'rootkit': 4,'ps': 4,'sqlattack': 4,'xterm': 4})
print(newlabeldf)

0         0
1         0
2         1
3         0
4         0
         ..
129082    4
129083    4
129084    4
129085    3
129086    3
Name: label, Length: 129087, dtype: int64


<ipython-input-12-8c472719a079>:1: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  newlabeldf=df_merged['label'].replace({ 'normal' : 0, 'neptune' : 1 ,'back': 1, 'land': 1, 'pod': 1, 'smurf': 1, 'teardrop': 1,'mailbomb': 1, 'apache2': 1, 'processtable': 1, 'udpstorm': 1, 'worm': 1,


In [ ]:
df_merged['label']=newlabeldf

In [ ]:
df_merged['label']

,label
0,0
1,0
2,1
3,0
4,0
...,...
129082,4
129083,4
129084,4
129085,3


In [ ]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping
from sklearn.model_selection import train_test_split
import numpy as np

In [ ]:
X=df_merged.drop('label', axis=1)
y=df_merged['label']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
X_train, X_val, y_train, y_val = train_test_split(X_train, y_train, test_size=0.2, random_state=42)

In [ ]:
model = Sequential([
    # Input Layer
    LSTM(10, activation='relu', return_sequences=True, input_shape=(24,1)),
    Dropout(0.1),

    # Hidden Layers
    LSTM(20, activation='relu', return_sequences=True),
    Dropout(0.2),

    LSTM(20, activation='relu'),
    Dropout(0.1),

    Dense(10, activation='relu'),

    # Output Layer
    Dense(5, activation='softmax')
])


In [ ]:
# Compile the model
model.compile(optimizer=Adam(learning_rate=0.001), loss='sparse_categorical_crossentropy', metrics=['accuracy'])

# Print model summary
model.summary()

Model: "sequential_6"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ lstm_18 (LSTM)                       │ (None, 24, 10)              │             480 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dropout_18 (Dropout)                 │ (None, 24, 10)              │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ lstm_19 (LSTM)                       │ (None, 24, 20)              │           2,480 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dropout_19 (Dropout)                 │ (None, 24, 20)              │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ lstm_20 (LSTM)                       │ (None, 20)                  │           3,280 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dropout_20 (Dropout)                 │ (None, 20)                  │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_12 (Dense)                     │ (None, 10)                  │             210 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_13 (Dense)                     │ (None, 5)                   │              55 │
└──────────────────────────────────────┴─────────────────────────────┴─────────────────┘

 Total params: 6,505 (25.41 KB)

 Trainable params: 6,505 (25.41 KB)

 Non-trainable params: 0 (0.00 B)

In [ ]:
# Early stopping callback
early_stopping = EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)

# Train the model
history = model.fit(X_train, y_train, epochs=50, batch_size=32, validation_data=(X_val, y_val), callbacks=[early_stopping])


Epoch 1/2
2582/2582 ━━━━━━━━━━━━━━━━━━━━ 98s 35ms/step - accuracy: 0.8011 - loss: 0.5802 - val_accuracy: 0.9232 - val_loss: 0.1744
Epoch 2/2
2582/2582 ━━━━━━━━━━━━━━━━━━━━ 143s 35ms/step - accuracy: 0.9308 - loss: 0.1905 - val_accuracy: 0.9450 - val_loss: 0.1753


In [ ]:
y_test.shape

(25818,)

In [ ]:
y_train

,label
43595,0
56301,1
68303,1
115169,2
113228,0
...,...
59447,2
10491,0
86371,1
43203,0
